# Velocity pipeline: steps 1 to 7

This simple notebook runs the velocity processing scripts in order:

1. Convert ROS bag velocity topics to CSV
2. Plot estimate and ground-truth velocity CSV files
3. Clean the CSV files
4. Align ground-truth velocity to the estimate timeline
5. Clip the evaluation window after liftoff, or use custom manual clipping
6. Bias-zero velocity errors and calculate RMSE
7. Plot absolute X/Y velocity error with estimate/GT velocity and local Z position


In [426]:
# Setup cell: change BAG_PATH if you want to process a different rosbag.
from pathlib import Path
import subprocess
import sys

BAG_PATH = Path("flight_2026-07-01-10-38-41.bag")

# Derived filenames used by scripts 1 to 6.
BASE = BAG_PATH.with_suffix("")
EST_CSV = Path(f"{BASE}_vel_est.csv")
GT_CSV = Path(f"{BASE}_vel_gt.csv")
EST_CLEAN_CSV = Path(f"{BASE}_vel_est_clean.csv")
GT_CLEAN_CSV = Path(f"{BASE}_vel_gt_clean.csv")
GT_ALIGNED_CSV = Path(f"{BASE}_vel_gt_clean_aligned.csv")
PLOTS_DIR = Path(f"plots_{BASE.name}")
EST_WINDOW_CSV = Path(f"{BASE}_vel_est_clean_window.csv")
GT_WINDOW_CSV = Path(f"{BASE}_vel_gt_clean_aligned_window.csv")
BIAS_ERRORS_CSV = Path(f"{EST_WINDOW_CSV.stem}_bias_errors{EST_WINDOW_CSV.suffix}")
ABS_ERROR_PLOTS_DIR = Path(f"plots_{BIAS_ERRORS_CSV.stem}")

def run_command(command):
    """Run one pipeline command and stop the notebook if it fails."""
    print("Running:", " ".join(str(part) for part in command))
    result = subprocess.run(command, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    result.check_returncode()


## 1. ROS bag to CSV

This reads the estimate and ground-truth velocity topics from the `.bag` file and saves two CSV files: one for estimated velocity and one for ground truth.

In [427]:
# Step 1: convert the rosbag into velocity CSV files.
run_command([
    "python3",
    "1_rosbag_to_csv_vel.py",
    "--bag",
    str(BAG_PATH),
])


Running: python3 1_rosbag_to_csv_vel.py --bag flight_2026-07-01-10-38-41.bag
Saved estimate velocity CSV to flight_2026-07-01-10-38-41_vel_est.csv
Estimate rows: 2759
Saved GT velocity CSV to flight_2026-07-01-10-38-41_vel_gt.csv
GT rows: 5545



## 2. Plot raw CSV data

This creates quick PNG plots for the raw estimate and ground-truth velocity data so you can visually inspect the signals before cleaning.

In [294]:
# Step 2: plot raw estimate and GT velocity CSV files.
run_command([
    "python3",
    "2_plot_vel_csv_data.py",
    "--est_csv",
    str(EST_CSV),
    "--gt_csv",
    str(GT_CSV),
    "--out_dir",
    str(PLOTS_DIR),
])


Running: python3 2_plot_vel_csv_data.py --est_csv flight_2026-07-01-10-25-49_vel_est.csv --gt_csv flight_2026-07-01-10-25-49_vel_gt.csv --out_dir plots_flight_2026-07-01-10-25-49


## 3. Clean velocity CSV files

This removes physically unreasonable velocity values, applies a sliding median filter, and fills gaps with linear interpolation.

In [428]:
# Step 3: clean both raw CSV files.
run_command([
    "python3",
    "3_vel_csv_dataset_cleaner.py",
    "--est_csv",
    str(EST_CSV),
    "--gt_csv",
    str(GT_CSV),
])


Running: python3 3_vel_csv_dataset_cleaner.py --est_csv flight_2026-07-01-10-38-41_vel_est.csv --gt_csv flight_2026-07-01-10-38-41_vel_gt.csv
Saved cleaned CSV to flight_2026-07-01-10-38-41_vel_est_clean.csv
Physical pruning counts:
  vel_x: 0
  vel_y: 0
  vel_z: 0
Median filter counts:
  vel_x: 0
  vel_y: 0
  vel_z: 0
Saved cleaned CSV to flight_2026-07-01-10-38-41_vel_gt_clean.csv
Physical pruning counts:
  vel_x: 0
  vel_y: 0
  vel_z: 0
Median filter counts:
  vel_x: 0
  vel_y: 6
  vel_z: 6



## 4. Align ground truth to estimate timeline

This compensates the ground-truth latency and resamples GT velocity onto the estimate timestamps using nearest-neighbor alignment.

In [429]:
# Step 4: align cleaned GT velocity to the cleaned estimate timeline.
run_command([
    "python3",
    "4_align_vel_csv.py",
    "--est_csv",
    str(EST_CLEAN_CSV),
    "--gt_csv",
    str(GT_CLEAN_CSV),
])


Running: python3 4_align_vel_csv.py --est_csv flight_2026-07-01-10-38-41_vel_est_clean.csv --gt_csv flight_2026-07-01-10-38-41_vel_gt_clean.csv
Saved aligned GT CSV to flight_2026-07-01-10-38-41_vel_gt_clean_aligned.csv
Estimate rows: 2759
GT rows (raw): 5545
Aligned rows: 2759



## 5A. Liftoff detection and window clipping

This detects liftoff from GT vertical velocity, then clips both estimate and aligned GT CSV files to the selected evaluation window.

In [23]:
# Step 5A: detect liftoff and clip the evaluation window.
# Change window_start/window_end below if you want a different time range after liftoff.
# run_command([
    # "python3",
    # "5_liftoff_and_window_clipping.py",
    # "--est_csv",
    # str(EST_CLEAN_CSV),
    # "--gt_csv",
    # str(GT_ALIGNED_CSV),
    # "--window_start",
    # "7.0",
    # "--window_end",
    # "22.0",
# ])


## 5B. Custom manual clipping

This clips both CSV files using manually chosen time values. Edit `WINDOW_START` and `WINDOW_END` directly in the code cell below. The values are seconds from the start of the aligned GT CSV.

In [430]:
# Step 5B: custom manual window clipping.
# Time values are seconds from the start of the aligned GT CSV.
WINDOW_START = 35.57
WINDOW_END = 38.57

# This writes the fixed *_window.csv files used by Step 6.
run_command([
    "python3",
    "5_manual_relative_window_clipping.py",
    "--est_csv",
    str(EST_CLEAN_CSV),
    "--gt_csv",
    str(GT_ALIGNED_CSV),
    "--window_start",
    str(WINDOW_START),
    "--window_end",
    str(WINDOW_END),
    "--output_suffix",
    "_window",
])


Running: python3 5_manual_relative_window_clipping.py --est_csv flight_2026-07-01-10-38-41_vel_est_clean.csv --gt_csv flight_2026-07-01-10-38-41_vel_gt_clean_aligned.csv --window_start 35.57 --window_end 38.57 --output_suffix _window
Input time ranges:
  est absolute: 1782902323.322249 -> 1782902377.840406
  gt  absolute: 1782902323.322249 -> 1782902377.840406
Requested clipping window:
  reference: aligned GT start
  reference absolute time: 1782902323.322249
  requested time values: 35.570000 -> 38.570000 s
  absolute: 1782902358.892249 -> 1782902361.892249
  requested duration: 3.000000 s
Actual clipped window:
  est: 35.577403 -> 38.567331 s
  est duration: 2.989927 s
  Note: est window is shorter than requested by 0.010073 s.
  gt : 35.577403 -> 38.567331 s
  gt  duration: 2.989927 s
  Note: gt  window is shorter than requested by 0.010073 s.
Saved estimate CSV to /home/tl/Downloads/Dataset_preparation_Airimu/velocity_evaluation/flight_2026-07-01-10-38-41_vel_est_clean_window.csv 

## 6. Bias zeroing and RMSE

This merges the selected windowed estimate and GT CSV files, subtracts the initial error as bias, and reports RMSE before and after bias correction.

In [431]:
# Step 6: compute velocity errors and RMSE from the fixed *_window.csv files.
# Run either Step 5A or Step 5B first to create these files.
print(f"Using estimate window: {EST_WINDOW_CSV}")
print(f"Using GT window: {GT_WINDOW_CSV}")

run_command([
    "python3",
    "6_bias_zeroing_and_rmse.py",
    "--est_csv",
    str(EST_WINDOW_CSV),
    "--gt_csv",
    str(GT_WINDOW_CSV),
])


Using estimate window: flight_2026-07-01-10-38-41_vel_est_clean_window.csv
Using GT window: flight_2026-07-01-10-38-41_vel_gt_clean_aligned_window.csv
Running: python3 6_bias_zeroing_and_rmse.py --est_csv flight_2026-07-01-10-38-41_vel_est_clean_window.csv --gt_csv flight_2026-07-01-10-38-41_vel_gt_clean_aligned_window.csv
Saved bias-normalized errors to /home/tl/Downloads/Dataset_preparation_Airimu/velocity_evaluation/flight_2026-07-01-10-38-41_vel_est_clean_window_bias_errors.csv (rows: 151)
RMSE (Not including bias): RMSE_x=0.271549, RMSE_y=0.186201, RMSE_z=0.125171, RMSE_xy=0.329256
RMSE (bias corrected): RMSE_x=0.068213, RMSE_y=0.051508, RMSE_z=0.067545, RMSE_xy=0.085476
Saved RMSE summary to /home/tl/Downloads/Dataset_preparation_Airimu/velocity_evaluation/flight_2026-07-01-10-38-41_vel_est_clean_window_rmse_summary.txt



## 7. Absolute velocity error with local Z position

This creates three high-resolution stacked plots: X velocity/error over local Z, Y velocity/error over local Z, and `sqrt(err_x^2 + err_y^2)` over local Z. The Z trace is clipped from `/mavros/local_position/pose/pose/position/z` using the same evaluation window.


In [432]:
# Step 7: plot absolute velocity errors and local Z position.
# Run Step 6 first so BIAS_ERRORS_CSV exists.
print(f"Using bias-error CSV: {BIAS_ERRORS_CSV}")
print(f"Using estimate window: {EST_WINDOW_CSV}")
print(f"Using GT window: {GT_WINDOW_CSV}")
print(f"Using rosbag: {BAG_PATH}")

run_command([
    "python3",
    "7_plot_absolute_velocity_error.py",
    "--bias_errors_csv",
    str(BIAS_ERRORS_CSV),
    "--est_csv",
    str(EST_WINDOW_CSV),
    "--gt_csv",
    str(GT_WINDOW_CSV),
    "--bag",
    str(BAG_PATH),
    "--out_dir",
    str(ABS_ERROR_PLOTS_DIR),
])


Using bias-error CSV: flight_2026-07-01-10-38-41_vel_est_clean_window_bias_errors.csv
Using estimate window: flight_2026-07-01-10-38-41_vel_est_clean_window.csv
Using GT window: flight_2026-07-01-10-38-41_vel_gt_clean_aligned_window.csv
Using rosbag: flight_2026-07-01-10-38-41.bag
Running: python3 7_plot_absolute_velocity_error.py --bias_errors_csv flight_2026-07-01-10-38-41_vel_est_clean_window_bias_errors.csv --est_csv flight_2026-07-01-10-38-41_vel_est_clean_window.csv --gt_csv flight_2026-07-01-10-38-41_vel_gt_clean_aligned_window.csv --bag flight_2026-07-01-10-38-41.bag --out_dir plots_flight_2026-07-01-10-38-41_vel_est_clean_window_bias_errors
Window: 1782902358.899652 -> 1782902361.889580 (2.989927 s)
Flight-relative window: 35.59 -> 38.58 s
Drift Rate: 0.049473 m/s^2
Drift Intercept: 0.251598 m/s
Saved Z window CSV to plots_flight_2026-07-01-10-38-41_vel_est_clean_window_bias_errors/flight_2026-07-01-10-38-41_local_position_z_window.csv (rows: 150)
Saved plot to plots_flight_20

Done. The main output files should be named like:

- `*_vel_est_clean_window.csv` and `*_vel_gt_clean_aligned_window.csv` for the selected clipping method
- `*_bias_errors.csv` and `*_rmse_summary.txt` for Step 6
- `*_abs_error_vel_x_with_z.png`, `*_abs_error_vel_y_with_z.png`, `*_xy_error_magnitude_with_z.png`, and `*_local_position_z_window.csv` for Step 7
